# 01 — Multi-Temporal SAR & Terrain Data Pipeline (Earthdata, AOI-Only Remote Slicing)

End-to-End-Pipeline zum Suchen, **direkten AOI-Zuschneiden via `earthaccess.open()`** und räumlich-zeitlichen Ausrichten (**Case 3: Multi-Temporal + Static Topography, ohne SME**) über **`earthaccess`** und das `EARTHDATA_TOKEN` aus `.env`.

* **Kein Full-Granule-Download für SAR:** Für `NISAR` (`.h5`) und `OPERA` (COGs) werden per HTTP-Range-Request **nur die Pixel innerhalb der `AOI_BBOX`** geladen (statt ~2–3,5 GB pro NISAR-Szene). Nur das kleine statische `NASADEM` (~7,5 MB `.zip`) wird lokal entpackt.
* **Gesammelte Datenquellen (kein `SME2`):**
  1. **NISAR L2 `GCOV`** (`NISAR_L2_GCOV_V1`, Fallback `NISAR_L2_GCOV_PROVISIONAL_V1`) — L-Band Polarimetrische Backscatter-Kovarianz (`HHHH`, `HVHV`, `VVVV`, ...)
  2. **NISAR L2 `GUNW`** (`NISAR_L2_GUNW_V1`, Fallback `NISAR_L2_GUNW_PROVISIONAL_V1`) — Interferometrische Kohärenz (`coherenceMagnitude`)
  3. **OPERA L2 `RTC-S1`** (`OPERA_L2_RTC-S1_V1`) — Sentinel-1 C-Band RTC Backscatter (`VV`, `VH`) für tiefe Zeitreihen ($T > 4$, ab 2016)
  4. **NASADEM** (`NASADEM_HGT`, v001) — Statisches DEM + abgeleitete `slope` & `aspect`
  5. *(Optional)* **OPERA L3 `DSWx-S1`** (`OPERA_L3_DSWX-S1_V1`) — Dynamische Oberflächenwasser-Referenzmasken


## 1. Setup & Earthdata-Authentifizierung

In [ ]:
import os
import re
import zipfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
import h5py
import rasterio
from rasterio.enums import Resampling
import xarray as xr
import rioxarray  # noqa: F401 (registriert .rio-Accessor)
from rioxarray.exceptions import NoDataInBounds
import earthaccess
import matplotlib.pyplot as plt

# Projektroot finden (funktioniert sowohl aus Repo-Root als auch aus notebooks/)
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), Path.cwd().parent] if (p / ".env").exists()),
    Path.cwd(),
)
ENV_PATH = PROJECT_ROOT / ".env"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# .env einlesen
if ENV_PATH.exists():
    for line in ENV_PATH.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip("\"'"))

if not os.environ.get("EARTHDATA_TOKEN"):
    raise ValueError("EARTHDATA_TOKEN nicht gefunden. Bitte in .env setzen.")

auth = earthaccess.login(strategy="environment")
print("Earthdata Login erfolgreich:", auth.authenticated)
print("Raw-Ordner (nur NASADEM-Cache):", RAW_DIR)
print("Processed-Ordner (AOI-Cubes):", PROCESSED_DIR)


## 2. Konfiguration: Region (AOI), Zeitintervall & Zielgitter (`T > 4`)

Passe `RUN_NAME`, `AOI_BBOX` (oder `AOI_GEOJSON`), `START_DATE` / `END_DATE` und `MAX_TIMESTEPS_PER_COLLECTION` an.
* Hinweis: `MAX_TIMESTEPS_PER_COLLECTION = None` lädt **alle** verfügbaren Zeitpunkte im Intervall. Mit einem Limit (z. B. `16`) bleiben erste Testläufe schnell, garantieren aber dennoch `T > 4`.


In [ ]:
# --- Benutzer-Konfiguration ---
RUN_NAME = "oberrhein_test"

# Optional: Pfad zu einer GeoJSON/Shapefile-Datei; wenn None, wird AOI_BBOX verwendet
AOI_GEOJSON = None
# (west, south, east, north) in EPSG:4326 — Beispiel: Oberrhein (Wald, Agrar, Wasser, Urban)
AOI_BBOX = (8.28, 49.02, 8.58, 49.28)

START_DATE = "2025-01-01"
END_DATE = "2026-11-14"

# Temporal-Anforderung aus Case 3: strikt T > 4
MIN_TIMESTEPS = 5
# None = alle verfügbaren Zeitpunkte laden; oder int (z. B. 16) für gleichmäßig verteilte Stichprobe
MAX_TIMESTEPS_PER_COLLECTION = 16

# Zielauflösung in Metern (UTM automatisch aus AOI geschätzt)
TARGET_RES_M = 30.0

# Welche Produkte sollen einbezogen werden? (SME wird bewusst NICHT verwendet)
INCLUDE_NISAR_GCOV = True
INCLUDE_NISAR_GUNW = True
INCLUDE_OPERA_RTC_S1 = True
INCLUDE_NASADEM = True
INCLUDE_OPERA_DSWX_S1 = False  # Optional als Wasser-Referenzlayer

# True = Remote-Slicing & Export ausführen; False = nur Suche/Auswahl prüfen
RUN_FETCH = True

if AOI_GEOJSON is not None:
    aoi_gdf = gpd.read_file(AOI_GEOJSON).to_crs(epsg=4326)
    AOI_BBOX = tuple(float(v) for v in aoi_gdf.total_bounds)
else:
    aoi_gdf = gpd.GeoDataFrame(geometry=[box(*AOI_BBOX)], crs="EPSG:4326")

TARGET_UTM_CRS = aoi_gdf.estimate_utm_crs()
OUT_DIR = PROCESSED_DIR / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"AOI BBOX (WGS84): {AOI_BBOX}")
print(f"Ziel-CRS (UTM):   {TARGET_UTM_CRS.to_string()} @ {TARGET_RES_M} m")
print(f"Zeitintervall:    {START_DATE} .. {END_DATE} (min T >= {MIN_TIMESTEPS})")


## 3. Multi-Dataset Suche via `earthaccess`

Für NISAR-Produkte wird zuerst die `V1` (Validated)-Collection abgefragt und um `PROVISIONAL_V1` ergänzt, falls für frühere 2026-Daten noch kein `V1`-Granule existiert. `NASADEM_HGT` ist statisch und wird **ohne** Zeitfilter gesucht.


In [ ]:
def _extract_granule_date(granule) -> str | None:
    umm = granule.get("umm", {})
    te = umm.get("TemporalExtent", {})
    rdt = te.get("RangeDateTime") or {}
    sdt = te.get("SingleDateTime")
    raw = rdt.get("BeginningDateTime") or sdt
    if not raw:
        return None
    return str(raw)[:10]


def search_collection(
    name: str,
    short_names: list[str],
    bbox: tuple[float, float, float, float],
    temporal: tuple[str, str] | None,
    version: str | None = None,
) -> list[dict]:
    records = []
    seen_native_ids = set()

    for sn in short_names:
        kwargs = {"short_name": sn, "bounding_box": bbox}
        if temporal is not None:
            kwargs["temporal"] = temporal
        if version is not None:
            kwargs["version"] = version

        granules = earthaccess.search_data(**kwargs)
        for g in granules:
            native_id = g.get("meta", {}).get("native-id", "")
            dedup_key = native_id.replace("_PROVISIONAL_", "_")
            if dedup_key in seen_native_ids:
                continue
            seen_native_ids.add(dedup_key)

            raw_size = g.size() if callable(getattr(g, "size", None)) else g.get("size", 0.0)
            records.append(
                {
                    "dataset": name,
                    "short_name": sn,
                    "granule_id": native_id,
                    "date": _extract_granule_date(g),
                    "size_mb": float(raw_size or 0.0),
                    "granule": g,
                }
            )
    return records


search_plan = []
if INCLUDE_NISAR_GCOV:
    search_plan.append(
        ("nisar_gcov", ["NISAR_L2_GCOV_V1", "NISAR_L2_GCOV_PROVISIONAL_V1"], (START_DATE, END_DATE), None)
    )
if INCLUDE_NISAR_GUNW:
    search_plan.append(
        ("nisar_gunw", ["NISAR_L2_GUNW_V1", "NISAR_L2_GUNW_PROVISIONAL_V1"], (START_DATE, END_DATE), None)
    )
if INCLUDE_OPERA_RTC_S1:
    search_plan.append(("opera_rtc_s1", ["OPERA_L2_RTC-S1_V1"], (START_DATE, END_DATE), None))
if INCLUDE_NASADEM:
    search_plan.append(("nasadem", ["NASADEM_HGT"], None, "001"))
if INCLUDE_OPERA_DSWX_S1:
    search_plan.append(("opera_dswx_s1", ["OPERA_L3_DSWX-S1_V1"], (START_DATE, END_DATE), None))

all_records = []
for name, short_names, temp_range, ver in search_plan:
    recs = search_collection(name, short_names, AOI_BBOX, temp_range, version=ver)
    all_records.extend(recs)

catalog_df = pd.DataFrame(all_records)
if catalog_df.empty:
    raise RuntimeError("Keine Granules für die gewählte AOI/Zeitspanne gefunden.")

summary_df = (
    catalog_df.groupby("dataset", as_index=False)
    .agg(
        granules=("granule_id", "count"),
        unique_dates=("date", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        full_granules_mb=("size_mb", "sum"),
    )
    .sort_values("dataset")
)
summary_df["meets_T_gt_4"] = np.where(
    summary_df["dataset"] == "nasadem",
    True,
    summary_df["unique_dates"] >= MIN_TIMESTEPS,
)
display(summary_df)


## 4. Zeitliche Auswahl (`T > 4`) & AOI-Only Datenplan

Für jede zeitliche Collection werden bis zu `MAX_TIMESTEPS_PER_COLLECTION` gleichmäßig über das Zeitintervall verteilte Aufnahmedaten ausgewählt. Nur `NASADEM` (~7,5 MB) wird lokal entpackt; alle SAR-Collections werden in Schritt 5 direkt per `earthaccess.open()` auf die `AOI_BBOX` zugeschnitten gestreamt.


In [ ]:
def select_dates_evenly(df: pd.DataFrame, max_dates: int | None) -> pd.DataFrame:
    if df.empty or max_dates is None:
        return df
    unique_dates = sorted(d for d in df["date"].dropna().unique())
    if len(unique_dates) <= max_dates:
        return df
    idx = np.linspace(0, len(unique_dates) - 1, max_dates, dtype=int)
    chosen = {unique_dates[i] for i in idx}
    return df[df["date"].isin(chosen)].copy()


selected_parts = []
for ds_name, grp in catalog_df.groupby("dataset"):
    if ds_name == "nasadem":
        selected_parts.append(grp)
    else:
        selected_parts.append(select_dates_evenly(grp, MAX_TIMESTEPS_PER_COLLECTION))

selected_df = pd.concat(selected_parts, ignore_index=True)

print("Ausgewählte Granules (werden nur im AOI-Ausschnitt gestreamt, außer NASADEM):")
display(
    selected_df.groupby("dataset", as_index=False).agg(
        granules=("granule_id", "count"),
        unique_dates=("date", "nunique"),
        full_granules_mb=("size_mb", "sum"),
    )
)

temporal_only = selected_df[selected_df["dataset"] != "nasadem"]
total_unique_dates = temporal_only["date"].nunique()
if total_unique_dates < MIN_TIMESTEPS:
    print(f"WARNUNG: Nur {total_unique_dates} zeitliche Datenpunkte gefunden (< {MIN_TIMESTEPS}). Zeitfenster erweitern!")
else:
    print(f"OK: {total_unique_dates} eindeutige Zeitpunkte ausgewählt (T > 4 erfüllt).")


## 5. Gemeinsames Referenzgitter & Remote AOI-Slicing (`earthaccess.open`)

Alle SAR-Granules werden über `earthaccess.open()` geöffnet und **vor dem Reprojizieren** auf die gepufferte `AOI_BBOX` zugeschnitten:
- **OPERA `RTC-S1` / `DSWx-S1` (COGs):** `rioxarray.open_rasterio(...)` + `.rio.clip_box(...)` liest nur die benötigten COG-Kacheln via HTTP-Range-Request.
- **NISAR `GCOV` / `GUNW` (HDF5):** Projiziert `AOI_BBOX` ins native Grid-CRS der HDF5-Datei und liest via `h5py` ausschließlich das Slice `[y0:y1, x0:x1]`.


In [ ]:
def build_target_grid(aoi_wgs84: gpd.GeoDataFrame, target_crs, res_m: float) -> xr.DataArray:
    aoi_utm = aoi_wgs84.to_crs(target_crs)
    minx, miny, maxx, maxy = aoi_utm.total_bounds
    minx = np.floor(minx / res_m) * res_m
    miny = np.floor(miny / res_m) * res_m
    maxx = np.ceil(maxx / res_m) * res_m
    maxy = np.ceil(maxy / res_m) * res_m

    xs = np.arange(minx + res_m / 2, maxx, res_m)
    ys = np.arange(maxy - res_m / 2, miny, -res_m)  # Nord-nach-Süd

    grid = xr.DataArray(
        np.zeros((len(ys), len(xs)), dtype=np.float32),
        coords={"y": ys, "x": xs},
        dims=("y", "x"),
    )
    grid = grid.rio.write_crs(target_crs)
    grid = grid.rio.set_spatial_dims(x_dim="x", y_dim="y")
    return grid


def align_to_target(da: xr.DataArray, target_grid: xr.DataArray, resampling=Resampling.bilinear) -> xr.DataArray:
    if da.rio.crs is None:
        raise ValueError("DataArray hat kein CRS.")
    da = da.rio.set_spatial_dims(x_dim="x", y_dim="y")
    return da.rio.reproject_match(target_grid, resampling=resampling)


def mosaic_aligned(parts: list[xr.DataArray], mode: str = "mean") -> xr.DataArray:
    if len(parts) == 1:
        return parts[0].astype("float32")
    if mode == "first":
        out = parts[0]
        for nxt in parts[1:]:
            out = out.combine_first(nxt)
        return out.astype("float32")
    stacked = xr.concat(parts, dim="_mosaic")
    return stacked.mean(dim="_mosaic", skipna=True).astype("float32")


def to_db(da: xr.DataArray) -> xr.DataArray:
    valid = da.where(da > 0)
    return (10.0 * np.log10(valid)).astype("float32")


def _filter_granule_urls(granules: list, suffixes: tuple[str, ...]) -> list[tuple[str, str]]:
    items = []
    for g in granules:
        dt = _extract_granule_date(g)
        direct_by_name = {
            u.rsplit("/", 1)[-1]: u for u in g.data_links(access="direct")
        }
        for url in g.data_links(access="external"):
            if not url.endswith(suffixes):
                continue
            # Ältere OPERA-Granules liefern datapool.asf.alaska.edu (OAuth/Basic),
            # während der zugehörige Earthdata-Cloud-Link Bearer-Token-Auth unterstützt.
            if "datapool.asf.alaska.edu" in url:
                fname = url.rsplit("/", 1)[-1]
                s3_url = direct_by_name.get(fname, "")
                if s3_url.startswith("s3://asf-cumulus-prod-opera-products/"):
                    url = s3_url.replace(
                        "s3://asf-cumulus-prod-opera-products/",
                        "https://cumulus.asf.earthdatacloud.nasa.gov/OPERA/",
                        1,
                    )
            items.append((dt, url))
    return items


def _clip_remote_cog(file_obj, aoi_wgs84: gpd.GeoDataFrame, target_grid: xr.DataArray, resampling=Resampling.bilinear) -> xr.DataArray | None:
    da = rioxarray.open_rasterio(file_obj, masked=True).squeeze("band", drop=True)
    if da.rio.crs is None:
        return None
    minx, miny, maxx, maxy = aoi_wgs84.to_crs(da.rio.crs).total_bounds
    pad = TARGET_RES_M * 4.0
    try:
        clipped = da.rio.clip_box(minx - pad, miny - pad, maxx + pad, maxy + pad)
    except NoDataInBounds:
        return None
    if clipped.size == 0:
        return None
    return align_to_target(clipped, target_grid, resampling=resampling)


def _read_nisar_epsg(group: h5py.Group) -> int:
    proj = group["projection"]
    epsg_attr = proj.attrs.get("epsg_code")
    if epsg_attr is not None:
        return int(epsg_attr)
    return int(proj[()])


def _slice_h5_datasets(
    grp: h5py.Group,
    dataset_paths: list[str],
    aoi_wgs84: gpd.GeoDataFrame,
    target_grid: xr.DataArray,
    resampling=Resampling.bilinear,
) -> dict[str, xr.DataArray]:
    epsg = _read_nisar_epsg(grp)
    xs = grp["xCoordinates"][:]
    ys = grp["yCoordinates"][:]

    minx, miny, maxx, maxy = aoi_wgs84.to_crs(epsg=epsg).total_bounds
    pad = TARGET_RES_M * 4.0
    xi = np.where((xs >= minx - pad) & (xs <= maxx + pad))[0]
    yi = np.where((ys >= miny - pad) & (ys <= maxy + pad))[0]
    if len(xi) == 0 or len(yi) == 0:
        return {}

    x0, x1 = int(xi[0]), int(xi[-1]) + 1
    y0, y1 = int(yi[0]), int(yi[-1]) + 1
    sub_x = xs[x0:x1].astype("float64")
    sub_y = ys[y0:y1].astype("float64")

    out = {}
    for dpath in dataset_paths:
        if dpath not in grp:
            continue
        arr = grp[dpath][y0:y1, x0:x1].astype("float32")
        da = xr.DataArray(arr, coords={"y": sub_y, "x": sub_x}, dims=("y", "x"))
        da = da.rio.write_crs(f"EPSG:{epsg}").rio.set_spatial_dims(x_dim="x", y_dim="y")
        out[dpath] = align_to_target(da, target_grid, resampling=resampling)
    return out


target_grid = build_target_grid(aoi_gdf, TARGET_UTM_CRS, TARGET_RES_M)
print(f"Target Grid Shape (y, x): {target_grid.shape}, CRS: {target_grid.rio.crs}")


In [ ]:
def fetch_nasadem_static(nasadem_df: pd.DataFrame, aoi_wgs84: gpd.GeoDataFrame, target_grid: xr.DataArray) -> xr.Dataset | None:
    if nasadem_df.empty:
        return None
    ds_dir = RAW_DIR / RUN_NAME / "nasadem"
    ds_dir.mkdir(parents=True, exist_ok=True)
    paths = earthaccess.download(nasadem_df["granule"].tolist(), local_path=str(ds_dir))

    hgt_files = []
    for p in paths:
        p = Path(p)
        if p.suffix.lower() == ".zip":
            with zipfile.ZipFile(p, "r") as zf:
                zf.extractall(ds_dir)
                hgt_files.extend(ds_dir / m for m in zf.namelist() if m.lower().endswith(".hgt"))
        elif p.suffix.lower() == ".hgt":
            hgt_files.append(p)

    parts = []
    for p in hgt_files:
        da = rioxarray.open_rasterio(p, masked=True).squeeze("band", drop=True)
        if da.rio.crs is None:
            da = da.rio.write_crs("EPSG:4326")
        minx, miny, maxx, maxy = aoi_wgs84.total_bounds
        da = da.rio.clip_box(minx - 0.01, miny - 0.01, maxx + 0.01, maxy + 0.01)
        parts.append(align_to_target(da, target_grid, resampling=Resampling.bilinear))

    if not parts:
        return None
    elev = mosaic_aligned(parts, mode="mean")
    dy_dx = np.gradient(elev.values, TARGET_RES_M, TARGET_RES_M)
    dz_dy, dz_dx = -dy_dx[0], dy_dx[1]
    slope_deg = np.degrees(np.arctan(np.hypot(dz_dx, dz_dy))).astype("float32")
    aspect_deg = (np.degrees(np.arctan2(dz_dy, -dz_dx)) % 360.0).astype("float32")

    return xr.Dataset(
        {
            "elevation_m": elev,
            "slope_deg": elev.copy(data=slope_deg),
            "aspect_deg": elev.copy(data=aspect_deg),
        }
    )


def fetch_opera_rtc_cube(df: pd.DataFrame, aoi_wgs84: gpd.GeoDataFrame, target_grid: xr.DataArray) -> xr.Dataset | None:
    if df.empty:
        return None
    pol_suffixes = ("_VV.tif", "_VH.tif", "_HH.tif", "_HV.tif")
    items = _filter_granule_urls(df["granule"].tolist(), pol_suffixes)
    if not items:
        return None

    urls = [u for _, u in items]
    print(f"OPERA RTC-S1: streame {len(urls)} COG-Bänder im AOI-Fenster...")
    open_files = earthaccess.open(urls, show_progress=True)

    by_date_pol: dict[str, dict[str, list[xr.DataArray]]] = defaultdict(lambda: defaultdict(list))
    for (dt, url), fobj in zip(items, open_files):
        pol = url.rsplit("_", 1)[-1].replace(".tif", "")
        aligned = _clip_remote_cog(fobj, aoi_wgs84, target_grid, resampling=Resampling.bilinear)
        if aligned is not None:
            by_date_pol[dt][pol].append(aligned)

    if not by_date_pol:
        return None
    dates = sorted(by_date_pol.keys())
    all_pols = sorted({pol for pmap in by_date_pol.values() for pol in pmap})
    ds_dict = {}
    for pol in all_pols:
        per_date = [
            to_db(mosaic_aligned(by_date_pol[dt][pol], mode="mean"))
            if by_date_pol[dt].get(pol)
            else xr.full_like(target_grid, np.nan, dtype="float32")
            for dt in dates
        ]
        ds_dict[f"{pol}_dB"] = xr.concat(per_date, dim=pd.Index(pd.to_datetime(dates), name="time"))

    ds = xr.Dataset(ds_dict)
    if "VH_dB" in ds and "VV_dB" in ds:
        ds["VH_minus_VV_dB"] = (ds["VH_dB"] - ds["VV_dB"]).astype("float32")
    return ds


def fetch_nisar_gcov_cube(df: pd.DataFrame, aoi_wgs84: gpd.GeoDataFrame, target_grid: xr.DataArray) -> xr.Dataset | None:
    if df.empty:
        return None
    items = _filter_granule_urls(df["granule"].tolist(), (".h5",))
    if not items:
        return None

    urls = [u for _, u in items]
    print(f"NISAR GCOV: streame {len(urls)} HDF5-Dateien im AOI-Slice...")
    open_files = earthaccess.open(urls, show_progress=True)

    candidate_terms = ["HHHH", "HVHV", "VVVV", "VHVH"]
    by_date_term: dict[str, dict[str, list[xr.DataArray]]] = defaultdict(lambda: defaultdict(list))

    for (dt, _), fobj in zip(items, open_files):
        with h5py.File(fobj, "r") as hf:
            freq_path = "/science/LSAR/GCOV/grids/frequencyA"
            if freq_path not in hf:
                freq_path = "/science/LSAR/GCOV/grids/frequencyB"
            if freq_path not in hf:
                continue
            sliced = _slice_h5_datasets(hf[freq_path], candidate_terms, aoi_wgs84, target_grid)
            for term, da in sliced.items():
                by_date_term[dt][term].append(da)

    if not by_date_term:
        return None
    dates = sorted(by_date_term.keys())
    present_terms = sorted({t for tmap in by_date_term.values() for t in tmap})
    ds_dict = {}
    for term in present_terms:
        per_date = [
            to_db(mosaic_aligned(by_date_term[dt][term], mode="mean"))
            if by_date_term[dt].get(term)
            else xr.full_like(target_grid, np.nan, dtype="float32")
            for dt in dates
        ]
        ds_dict[f"{term}_dB"] = xr.concat(per_date, dim=pd.Index(pd.to_datetime(dates), name="time"))
    return xr.Dataset(ds_dict)


def fetch_nisar_gunw_cube(df: pd.DataFrame, aoi_wgs84: gpd.GeoDataFrame, target_grid: xr.DataArray) -> xr.Dataset | None:
    if df.empty:
        return None
    items = _filter_granule_urls(df["granule"].tolist(), (".h5",))
    if not items:
        return None

    urls = [u for _, u in items]
    print(f"NISAR GUNW: streame {len(urls)} HDF5-Dateien im AOI-Slice...")
    open_files = earthaccess.open(urls, show_progress=True)

    coh_paths = [f"{pol}/coherenceMagnitude" for pol in ("HH", "VV", "HV", "VH")]
    by_date_pol: dict[str, dict[str, list[xr.DataArray]]] = defaultdict(lambda: defaultdict(list))

    for (dt, _), fobj in zip(items, open_files):
        with h5py.File(fobj, "r") as hf:
            base = "/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram"
            if base not in hf:
                continue
            sliced = _slice_h5_datasets(hf[base], coh_paths, aoi_wgs84, target_grid)
            for cpath, da in sliced.items():
                pol = cpath.split("/", 1)[0]
                by_date_pol[dt][pol].append(da)

    if not by_date_pol:
        return None
    dates = sorted(by_date_pol.keys())
    present_pols = sorted({pol for pmap in by_date_pol.values() for pol in pmap})
    ds_dict = {}
    for pol in present_pols:
        per_date = [
            mosaic_aligned(by_date_pol[dt][pol], mode="mean")
            if by_date_pol[dt].get(pol)
            else xr.full_like(target_grid, np.nan, dtype="float32")
            for dt in dates
        ]
        ds_dict[f"coherence_{pol}"] = xr.concat(per_date, dim=pd.Index(pd.to_datetime(dates), name="time"))
    return xr.Dataset(ds_dict)


def fetch_opera_dswx_cube(df: pd.DataFrame, aoi_wgs84: gpd.GeoDataFrame, target_grid: xr.DataArray) -> xr.Dataset | None:
    if df.empty:
        return None
    items = _filter_granule_urls(df["granule"].tolist(), ("_WTR.tif", "_BWTR.tif"))
    if not items:
        return None

    urls = [u for _, u in items]
    print(f"OPERA DSWx-S1: streame {len(urls)} Wasser-COGs im AOI-Fenster...")
    open_files = earthaccess.open(urls, show_progress=True)

    by_date: dict[str, list[xr.DataArray]] = defaultdict(list)
    for (dt, _), fobj in zip(items, open_files):
        aligned = _clip_remote_cog(fobj, aoi_wgs84, target_grid, resampling=Resampling.nearest)
        if aligned is not None:
            by_date[dt].append(aligned)

    if not by_date:
        return None
    dates = sorted(by_date.keys())
    per_date = [mosaic_aligned(by_date[dt], mode="first") for dt in dates]
    return xr.Dataset({"dswx_wtr": xr.concat(per_date, dim=pd.Index(pd.to_datetime(dates), name="time"))})


## 6. Remote-Extraktion, Export (NetCDF + Manifest) & Qualitäts-Check

Jeder auf die AOI zugeschnittene Datencube wird komprimiert unter `data/processed/<RUN_NAME>/` gespeichert und im `manifest.csv` dokumentiert (inkl. `T` und gültiger Pixel-Abdeckung in %).


In [ ]:
datasets_out = {}

if RUN_FETCH:
    if INCLUDE_NASADEM:
        datasets_out["nasadem_static"] = fetch_nasadem_static(
            selected_df[selected_df["dataset"] == "nasadem"], aoi_gdf, target_grid
        )
    if INCLUDE_OPERA_RTC_S1:
        datasets_out["opera_rtc_s1"] = fetch_opera_rtc_cube(
            selected_df[selected_df["dataset"] == "opera_rtc_s1"], aoi_gdf, target_grid
        )
    if INCLUDE_NISAR_GCOV:
        datasets_out["nisar_gcov"] = fetch_nisar_gcov_cube(
            selected_df[selected_df["dataset"] == "nisar_gcov"], aoi_gdf, target_grid
        )
    if INCLUDE_NISAR_GUNW:
        datasets_out["nisar_gunw"] = fetch_nisar_gunw_cube(
            selected_df[selected_df["dataset"] == "nisar_gunw"], aoi_gdf, target_grid
        )
    if INCLUDE_OPERA_DSWX_S1:
        datasets_out["opera_dswx_s1"] = fetch_opera_dswx_cube(
            selected_df[selected_df["dataset"] == "opera_dswx_s1"], aoi_gdf, target_grid
        )
else:
    print("RUN_FETCH=False: Keine Remote-Daten abgerufen.")

manifest_rows = []
for name, ds in datasets_out.items():
    if ds is None:
        print(f"[SKIP] {name}: keine verwertbaren Layer in der AOI.")
        continue

    out_nc = OUT_DIR / f"{name}.nc"
    encoding = {var: {"zlib": True, "complevel": 4} for var in ds.data_vars}
    ds.to_netcdf(out_nc, engine="h5netcdf", encoding=encoding)

    rel_file = str(out_nc.relative_to(PROJECT_ROOT)) if out_nc.is_relative_to(PROJECT_ROOT) else str(out_nc)
    first_var = next(iter(ds.data_vars))
    if "time" in ds.dims:
        for t in ds["time"].values:
            slice_arr = ds[first_var].sel(time=t).values
            valid_pct = float(np.isfinite(slice_arr).mean() * 100.0)
            manifest_rows.append(
                {
                    "dataset": name,
                    "date": str(pd.Timestamp(t).date()),
                    "variables": ",".join(ds.data_vars),
                    "valid_pct": round(valid_pct, 2),
                    "file": rel_file,
                }
            )
    else:
        valid_pct = float(np.isfinite(ds[first_var].values).mean() * 100.0)
        manifest_rows.append(
            {
                "dataset": name,
                "date": "static",
                "variables": ",".join(ds.data_vars),
                "valid_pct": round(valid_pct, 2),
                "file": rel_file,
            }
        )
    print(f"[OK] {name} -> {out_nc} | dims={dict(ds.sizes)}")

manifest_df = pd.DataFrame(manifest_rows)
if not manifest_df.empty:
    manifest_df.to_csv(OUT_DIR / "manifest.csv", index=False)
    display(manifest_df)
    temporal_T = manifest_df[manifest_df["date"] != "static"]["date"].nunique()
    print(f"Gesamte eindeutige Zeitpunkte im Export: T = {temporal_T} (Anforderung T > 4: {temporal_T > 4})")


## 7. Visueller Sanity-Check

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

if datasets_out.get("nasadem_static") is not None:
    datasets_out["nasadem_static"]["elevation_m"].plot(ax=axes[0], cmap="terrain")
    axes[0].set_title("NASADEM Elevation (m)")
else:
    axes[0].set_title("NASADEM (nicht geladen)")

if datasets_out.get("opera_rtc_s1") is not None and "VV_dB" in datasets_out["opera_rtc_s1"]:
    ds_s1 = datasets_out["opera_rtc_s1"]
    ds_s1["VV_dB"].isel(time=0).plot(ax=axes[1], cmap="gray", vmin=-25, vmax=0)
    axes[1].set_title(f"OPERA S1 VV dB ({str(ds_s1.time.values[0])[:10]})")
else:
    axes[1].set_title("OPERA S1 (nicht geladen)")

if datasets_out.get("nisar_gcov") is not None:
    ds_n = datasets_out["nisar_gcov"]
    v0 = next(iter(ds_n.data_vars))
    ds_n[v0].isel(time=0).plot(ax=axes[2], cmap="gray", vmin=-25, vmax=0)
    axes[2].set_title(f"NISAR GCOV {v0} ({str(ds_n.time.values[0])[:10]})")
else:
    axes[2].set_title("NISAR GCOV (nicht geladen)")

plt.tight_layout()
plt.show()
